# Chosen Model Analysis

This notebook is run only after the final comparison, portfolio robustness, and paired tests. It does not train or select a model. `DEEPSET_40_DYNAMIC` is fixed before these analyses based on the completed OOS comparison and the project's market-representation research objective.

Design decisions: regimes use trailing 12-month market volatility known at the signal date and an expanding historical median threshold shifted by one month; feature importance uses grouped OOS permutation importance from each matching annual DeepSets checkpoint, jointly permuting an underlying characteristic's current, lagged and velocity coordinates within a month; factor alpha uses FF5 plus momentum with Newey-West-6 inference; long and short legs are reported separately; cost discussion uses the already-generated 10%, 25%, and 50 bps robustness scenarios.

In [ ]:
from pathlib import Path
import os
import sys
import importlib

COLAB_PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/FDS Project')
WINDOWS_PROJECT_DIR = Path('C:/Users/sandh/OneDrive/Documents/Coding/FDS Project')
if Path('/content').exists() and not COLAB_PROJECT_DIR.exists():
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT_DIR = COLAB_PROJECT_DIR if COLAB_PROJECT_DIR.exists() else WINDOWS_PROJECT_DIR
os.chdir(PROJECT_DIR)
project_path = str(PROJECT_DIR)
sys.path = [path for path in sys.path if path != project_path]
sys.path.insert(0, project_path)
for module_name in tuple(sys.modules):
    if module_name == 'src' or module_name.startswith('src.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
print('Project directory:', PROJECT_DIR)

In [ ]:
from src.config import ExperimentConfig

CHOSEN_MODEL_ID = 'DEEPSET_40_DYNAMIC'  # Frozen from the completed OOS comparison.
DATA_PATH = PROJECT_DIR / 'jkp_USA_100chars_1980_2024.parquet'
OUTPUT_DIR = PROJECT_DIR / 'model_runs'
CONFIG = ExperimentConfig(
    experiment_id='core20_benchmarks_v1',
    project_dir=PROJECT_DIR, data_path=DATA_PATH, output_dir=OUTPUT_DIR,
    selected_models=(CHOSEN_MODEL_ID,), seed=42, use_gpu=False,
)
CONFIG.validate()

# This analysis needs the chosen model's transaction-cost and universe-robustness
# summary. Build it from the already-saved pooled predictions if it is absent or
# stale. This does not load model weights, retrain, or change predictions.
from src.portfolio_robustness import run_portfolio_robustness
chosen_robustness = run_portfolio_robustness(
    CONFIG.run_dir, model_ids=(CHOSEN_MODEL_ID,)
)

from src.chosen_model_analysis import validate_chosen_model_artifacts
chosen_artifacts = validate_chosen_model_artifacts(CONFIG, CHOSEN_MODEL_ID)
print('Chosen model:', CHOSEN_MODEL_ID)
print('Model signature:', chosen_artifacts['model_signature'])
print('Run directory:', CONFIG.run_dir)

## 1. Signal-date market regimes

In [ ]:
from src.chosen_model_analysis import build_signal_date_regimes, regime_stability

regimes = build_signal_date_regimes(CONFIG)
regime_results = regime_stability(CONFIG, CHOSEN_MODEL_ID, regimes)
display(regime_results)

In [ ]:
import matplotlib.pyplot as plt
regime_results.set_index('regime')[['annualized_return', 'sharpe']].plot.bar(
    subplots=True, layout=(1, 2), figsize=(10, 4), legend=False,
    title=['Annualized return', 'Sharpe ratio']
)
plt.suptitle(f'{CHOSEN_MODEL_ID}: stability across signal-date volatility regimes')
plt.tight_layout(); plt.show()

## 2. Feature importance by regime

This section uses each matching annual DeepSets OOS checkpoint. For every underlying characteristic, its current, lagged and velocity coordinates are permuted together across firms within each month. The increase in OOS MSE measures model reliance without refitting or changing model selection.

In [ ]:
from src.chosen_model_analysis import feature_importance_by_regime

importance = feature_importance_by_regime(CONFIG, CHOSEN_MODEL_ID, regimes)
top_importance = importance.query('importance_rank <= 10')
display(top_importance)

In [ ]:
pivot = top_importance.pivot(index='feature', columns='regime', values='importance_share').fillna(0)
pivot.plot.barh(figsize=(9, 7), title='Top grouped OOS permutation importance by regime')
plt.xlabel('Share of positive permutation-MSE increase'); plt.tight_layout(); plt.show()

## 3. FF5 plus momentum factor decomposition

This cell downloads monthly US factors from Kenneth French's official data library. Internet access is required only for this cell; the downloaded series enter the ex-post portfolio attribution, not model training or model selection.

In [ ]:
from src.chosen_model_analysis import download_ff5_momentum, factor_decomposition

factors = download_ff5_momentum()
factor_results = factor_decomposition(CONFIG, CHOSEN_MODEL_ID, factors)
display(factor_results)

## 4. Long-versus-short attribution and final cost discussion

In [ ]:
from IPython.display import Markdown, display
from src.chosen_model_analysis import long_short_and_cost_attribution

attribution, cost_discussion = long_short_and_cost_attribution(CONFIG, CHOSEN_MODEL_ID)
display(attribution)
display(Markdown(cost_discussion))

## 5. Final audit, frozen manifest, and report-ready outputs

This final step verifies the saved research artifacts, records the exact chosen model/data/code environment, and writes the audit, manifest, tables, and figures used for the report. It does not train or alter any model predictions.

In [ ]:
import pandas as pd
from src.project_finalization import (
    run_final_project_audit, write_frozen_manifest, generate_report_outputs,
)

FINAL_MODEL_IDS = (
    'LASSO_20', 'LGBM_20', 'XGBOOST_20', 'NN2_20', 'NN2_40', 'NN3_20', 'NN4_20', 'NN4_40',
    'LGBM_40', 'LGBM_60', 'LGBM_80', 'LGBM_100',
    'LGBM_20_LAG1', 'LGBM_20_LAG2',
    'LGBM_40_LAG1', 'LGBM_40_LAG2',
    'MLP_40', 'DEEPSET_40',
    'DEEPSET_40_LAG1', 'DEEPSET_40_DYNAMIC',
    'DEEPSET_20', 'DEEPSET_20_LAG1', 'DEEPSET_20_DYNAMIC',
    'HYBRID_LGBM20_DEEPSET20',
    'HYBRID_MLP40_DEEPSET40',
)
final_audit = run_final_project_audit(CONFIG, FINAL_MODEL_IDS, CHOSEN_MODEL_ID)
display(final_audit.loc[~final_audit['passed']])
if not final_audit['passed'].all():
    raise RuntimeError('Final project audit has failed checks; review the table above.')
frozen_manifest = write_frozen_manifest(CONFIG, FINAL_MODEL_IDS, CHOSEN_MODEL_ID)
report_files = generate_report_outputs(CONFIG, CHOSEN_MODEL_ID)
print('Final audit passed:', len(final_audit), 'checks')
print('Frozen chosen model:', frozen_manifest['chosen_model_id'])
display(pd.DataFrame({'report_file': list(report_files)}))

## Interpretation checklist

- Does performance and rank IC remain positive in both volatility regimes?
- Do the leading grouped permutation-importance characteristics remain economically interpretable across regimes?
- Is FF5+momentum alpha positive with a reliable HAC t-stat?
- Is performance balanced across the long and short legs?
- Does the strategy remain attractive under the 25 bps scenario, outside microcaps, and under the adverse missing-return stress?
- State explicitly that model selection preceded this analysis; these results explain and stress-test the chosen model rather than reopen the model search.